# Possesso — pipeline della run

**Caricamento → audit → palla → giocatori → distanze e possesso → output.**

Modifica solo CONFIG, poi usa **Restart & Run All**. Tutti i grafici mostrano la run selezionata: nessuna demo separata. I materiali e gli output reali restano locali, esclusi da Git. Nessun training/tuning sulla partita, confronto delle predizioni con il video o correzione manuale.

Le impostazioni iniziali caricano dieci minuti dall'inizio del gioco. I frame sono indici della registrazione, con fine esclusa. Le soglie sperimentali non sono validate né da ottimizzare su questa partita.

In [ ]:
CONFIG = {
    "source": "real",                 # "real"; "synthetic" solo per sviluppo senza dati
    "start_frame": 11185,
    "end_frame": 26185,                # None = fino alla fine della registrazione
    "ball_method": "none",            # "none", "linear", "velocity"
    "ball_max_gap_s": 0.5,             # limite sperimentale, non calibrato
    "player_policy": "none",          # "none", "nearest22"
    "show_plots": True,
    "preview_frames": 250,             # solo ampiezza del grafico di un buco
    "save_outputs": False,
    "run_name": "prima_run",
    "synthetic_duration_s": 120,        # usato soltanto con source="synthetic"
}

## 1. Caricamento — `src/data.py`, `src/runs.py`

Leggiamo solo tracking tattico e metadati. Il frame ATD parte da 1: il lettore sottrae uno. Le coordinate normalizzate vengono convertite in metri una sola volta. L'intervallo selezionato mantiene gli indici originali. Ogni traccia appartiene a una squadra, ma non identifica stabilmente una persona.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.runs import load_run
from src.notebook_io import audit_run, export_run
from src.review import stage_summary, plot_input, plot_ball_gap, plot_players_change, plot_decisions

raw, expected_rows = load_run(CONFIG, ROOT)
print("Fonte:", CONFIG["source"], "— frame:", len(raw), "— intervallo:", raw.frame.iloc[0], "–", raw.frame.iloc[-1])

## 2. Audit e preparazione — `src/checks.py`, `src/preparation.py`

Controlliamo numero, ordine e duplicati dei frame. Se l'indice è ambiguo, ci fermiamo. I grafici mostrano copertura della palla e quantità di tracce nell'intervallo corrente, senza spazio vuoto prima della run.

Creiamo una copia di lavoro: normalizziamo osservazioni non finite senza cambiare coordinate valide, riordinare o eliminare frame. Il lettore gestisce già molte assenze: un registro con zero modifiche è normale. Non eliminiamo coordinate fuori campo o possibili doppie tracce senza una regola esplicita.

In [ ]:
from src.preparation import prepare_tracking

audit = audit_run(raw, expected_rows, CONFIG["start_frame"])
prepared, cleaning_log = prepare_tracking(raw)
logs = {"preparazione": cleaning_log}
display(stage_summary(audit))
if CONFIG["show_plots"]:
    plot_input(raw)
    plt.show()

## 3. Palla mancante — `src/ball.py`, `src/reconstruction.py`

`none` lascia i buchi. `linear` collega due osservazioni ai lati di un buco breve; usa quindi anche il futuro. `velocity` prolunga il movimento delle due osservazioni precedenti. Buchi oltre `ball_max_gap_s` rimangono sconosciuti; non attraversiamo i confini dei tempi o salti nei frame.

`ball_xy` rimane originale; `ball_estimated_xy` contiene solo stime; `ball_used_xy` sceglie osservazione, altrimenti stima disponibile. La provenienza rimane registrata: una stima non è una rilevazione.

**Grafico:** primo buco ricostruito, altrimenti primo buco ancora sconosciuto. Nero = osservazioni, croci arancioni = stime, fascia grigia = buco. Se è troppo lungo, mostriamo soltanto la finestra di anteprima: non inventiamo la traiettoria reale né un errore rispetto a una verità sconosciuta.

In [ ]:
from src.ball import prepare_ball

ball_tracking, ball_log = prepare_ball(prepared, CONFIG["ball_method"], CONFIG["ball_max_gap_s"], CONFIG["source"])
logs["palla"] = ball_log
display(stage_summary(ball_log))
if CONFIG["show_plots"]:
    plot_ball_gap(ball_tracking, CONFIG["preview_frames"])
    plt.show()

## 4. Giocatori — `src/players.py`

`none` non rimuove tracce. `nearest22` conserva le 22 più vicine alla **palla osservata** nei soli frame con più di 22 tracce. Se manca la palla, non interviene neanche se è stata stimata. Non aggiunge giocatori mancanti e non impone 11 per squadra.

È un filtro grezzo, non una deduplicazione: può eliminare persone reali lontane e conservare doppie tracce vicine.

**Grafico:** conteggio prima/dopo e primo frame modificato, con croci sulle osservazioni rimosse. Se nulla cambia, viene dichiarato; non mostriamo una modifica inesistente.

In [ ]:
from src.players import filter_players

tracking, player_log = filter_players(ball_tracking, CONFIG["player_policy"])
logs["giocatori"] = player_log
display(stage_summary(player_log))
if CONFIG["show_plots"]:
    plot_players_change(ball_tracking, tracking)
    plt.show()

## 5. Distanze e possesso — `src/features.py`, `src/algorithm.py`

Per ogni frame calcoliamo la distanza minima dalla palla operativa per ciascuna squadra. Vince la squadra con la distanza minore: H per casa, A per ospiti. Se manca una distanza o c'è parità, restituiamo X. Non riconosciamo D e non distinguiamo ancora controllo effettivo e vicinanza.

Le stime entrano nelle distanze se il modulo palla è attivo. Conserviamo `ball_source`, ma non applichiamo ancora un modello di fiducia alle stime.

**Grafico:** distanze effettivamente usate e sequenza di etichette, con tacche sui frame che usano una palla stimata. Il riepilogo confronta solo l'effetto della ricostruzione mantenendo gli stessi giocatori: meno X non dimostra maggiore accuratezza.

In [ ]:
from src.features import build_features
from src.algorithm import predict

features = build_features(tracking, ball_column="ball_used_xy")
predictions = predict(features)
observed_only = predict(build_features(tracking))
logs["effetto_palla"] = {
    "etichette_cambiate": int(predictions.label.ne(observed_only.label).sum()),
    "X_solo_osservazioni": int(observed_only.label.eq("X").sum()),
    "X_run_corrente": int(predictions.label.eq("X").sum()),
}
display(stage_summary(logs["effetto_palla"]))
if CONFIG["show_plots"]:
    plot_decisions(features, predictions)
    plt.show()

## 6. Output e salvataggio — `src/notebook_io.py`

Controlliamo formato, etichette e indice. Con `save_outputs=True`, scriviamo in `outputs/<run_name>/`:

- `predictions_slice.csv` per un intervallo, oppure `submission_baseline.csv` per la registrazione reale completa;
- `run.json`: configurazione, audit e operazioni dei moduli;
- `features.csv`: distanze e provenienza;
- `ball_provenance.csv`: coordinate originali, stimate e usate.

Lo stesso nome di run sovrascrive i file. Un CSV strutturalmente valido non dimostra accuratezza e la baseline non è ancora un riconoscitore completo del possesso. Prima di condividere il notebook, cancellare tutti gli output reali.

In [ ]:
output_checks = export_run(CONFIG, ROOT, tracking, features, predictions, audit, logs)
display(stage_summary(output_checks))
display(predictions.head(10))